# p109 blow-up / instability — is the ~27k event a known phenomenon?

`p109_seed485_dseed598` grokked early, sat stable ~5k–26k, then **spontaneously destabilized at ~27k** (test loss ~1e-7 → ~3) and re-cohered by ~30k. The companion notebook [`p109_late_reorganization.ipynb`](./p109_late_reorganization.ipynb) worked this event through MIScope's *own* instruments — parameter PCA, activation-DMD eigenvalues, per-neuron displacement — and found ~11 neurons exploding (one ~800× in activation magnitude) while the committed frequencies (4/14/27) survived.

This notebook asks a different question: **is this a named, well-studied phenomenon, seen from another angle?** Rather than our geometry lenses, it reaches for the optimizer- and attention-dynamics literature, where late-training spikes in a grokked model have established signatures.

Three literature threads converge on this kind of event:

1. **The slingshot mechanism** (Thilak et al., 2022). With adaptive optimizers (Adam), late training exhibits cyclic *slingshots*: the **last-layer weight norm** grows, then snaps, each snap coinciding with a training-loss spike — and grokking often begins at the first slingshot. The diagnostic is the last-layer weight norm.
2. **Attention entropy collapse** (Zhai et al., 2023, σReparam). A transformer instability in which attention sharpens toward one-hot — its entropy collapses — heralded by the **max (pre-softmax) attention logit** growing large. The diagnostic is the max attention logit.
3. **Parameter / neuron norm growth** (e.g. Merrill et al. on parameter-norm growth; outlier-feature work). Individual units' weights and activations diverge. Notebook 1's ~800× activation blow-up is the per-unit face of this.

**Working hypothesis:** the ~27k event is a *late, isolated slingshot* — a sudden last-layer-norm excursion — co-timed with a transient attention sharpening (a max-logit spike), with the neuron blow-up as the per-unit shadow of the same norm excursion. If so, p109's event is not exotic; it is a known instability arriving once, long after the model had apparently settled.

**First test (this notebook):** put the **last-layer weight norm** and the **max attention logit** on the same epoch axis. A slingshot predicts both spike together in the ~27k window.

---
*Conventions follow notebook 1: all data access goes through the miscope API (`variant.artifacts`, `variant.run_with_cache`), never file paths. Last-layer weight norm = Frobenius norm of the unembedding `W_U` (the final learnable map to logits); the MLP output projection `W_out` is the other candidate and is checked in the open threads. Max attention logit = max over the full (a,b) grid, all heads and query/key positions, of the pre-softmax scores at `blocks.0.attn.hook_attn_scores`.*

In [1]:
import os
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

# Locate the repo root (holds data/) and chdir there so relative data paths
# resolve from any launch directory.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
from miscope.families.discovery import load_family_from_dir

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
variant = fam.get_variant(prime=109, seed=485, data_seed=598)
PRIME = 109
GRID = [[a, b] for a in range(PRIME) for b in range(PRIME)]  # full (a,b) probe
variant

/home/megano/projects/mechinterp/miscope/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Variant(family='modulo_addition_1layer', name='p109_seed485_dseed598', state=analyzed)

## 1. Slingshot co-diagnostic — last-layer weight norm + max attention logit

The two literature signals on one epoch axis. `W_U` is read straight from `parameter_snapshot` (cheap). The max attention logit needs a forward pass per checkpoint, so we run the full (a,b) grid through each saved model and take the maximum pre-softmax score over all heads and query/key positions.

In [2]:
epochs = variant.artifacts.get_epochs("parameter_snapshot")


def last_layer_norm(epoch):
    """Frobenius norm of the unembedding W_U — the slingshot 'last-layer weight norm'."""
    return float(np.linalg.norm(variant.artifacts.load_epoch("parameter_snapshot", epoch)["W_U"]))


def max_attn_logit(epoch):
    """Max pre-softmax attention score over the full grid — the entropy-collapse signal."""
    _, cache = variant.run_with_cache(variant.make_probe(GRID), epoch=epoch)
    return float(cache["blocks.0.attn.hook_attn_scores"].max())


ll_norm = np.array([last_layer_norm(e) for e in epochs])
attn_max = np.array([max_attn_logit(e) for e in epochs])
print(f"computed {len(epochs)} epochs, {epochs[0]}..{epochs[-1]}")

computed 353 epochs, 0..34999


In [3]:
fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_trace(go.Scatter(x=epochs, y=ll_norm, mode="lines", name="last-layer norm ||W_U||_F",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=epochs, y=attn_max, mode="lines", name="max attention logit",
                         line=dict(color="#d62728"), yaxis="y2"))
fig.update_layout(
    title="p109 slingshot co-diagnostic: last-layer weight norm vs max attention logit",
    xaxis_title="epoch", yaxis_title="||W_U||_F",
    yaxis2=dict(title="max attention logit", overlaying="y", side="right"),
    legend=dict(x=0.01, y=0.99), height=460)
fig.show()

In [4]:
# Where does each signal peak, and do they co-time with the event window?
def peak(arr):
    i = int(np.argmax(arr))
    return epochs[i], float(arr[i])


e_norm, v_norm = peak(ll_norm)
e_attn, v_attn = peak(attn_max)
base = (np.array(epochs) >= 20000) & (np.array(epochs) <= 26000)  # settled-plateau reference
print(f"last-layer norm : peak {v_norm:.2f} at epoch {e_norm}  "
      f"(plateau median {np.median(ll_norm[base]):.2f}, ratio {v_norm/np.median(ll_norm[base]):.2f}x)")
print(f"max attn logit  : peak {v_attn:.2f} at epoch {e_attn}  "
      f"(plateau median {np.median(attn_max[base]):.2f}, ratio {v_attn/np.median(attn_max[base]):.2f}x)")
ev = (np.array(epochs) >= 26900) & (np.array(epochs) <= 30000)
print(f"\nin-event maxima: ||W_U|| {ll_norm[ev].max():.2f} @ {epochs[int(np.where(ev)[0][np.argmax(ll_norm[ev])])]}, "
      f"attn-logit {attn_max[ev].max():.2f} @ {epochs[int(np.where(ev)[0][np.argmax(attn_max[ev])])]}")

last-layer norm : peak 18.40 at epoch 2200  (plateau median 11.55, ratio 1.59x)
max attn logit  : peak 17.67 at epoch 200  (plateau median 2.37, ratio 7.47x)

in-event maxima: ||W_U|| 11.47 @ 26900, attn-logit 3.12 @ 28100


## 2. Localizing the diagnostics — does the blow-up surface in `W_out` and at the `=` query?

§1 showed the event is invisible to the *global* last-layer norm and the *global* max attention logit. The literature lenses are right; the **resolution** was wrong. Two localized re-runs, aimed where notebook 1 says the action is:

- **The per-neuron see-saw `||W_in[j]||` vs `||W_out[j]||`.** The ~11 exploders blow up in `W_in` (notebook 1 §1) and in activation magnitude (~800×, §7). For the function to survive — frequencies maintained, residual contribution bounded — the *output* row each exploder writes through should **down-scale** to compensate (`residual contribution = a_j · W_out[j]`). So the prediction is a see-saw: `||W_in[327]||` **up**, `||W_out[327]||` **down**, across the same window where the global `||W_out||_F` stays flat. That flat aggregate is the §1 wash-out; the per-row drop is the signal it hid.
- **Attention entropy at the `=` query position.** The cleaner entropy-collapse statement than the global max logit: at the `=` token (query index 2), how concentrated is attention over (a, b, =)? The mild, *lagging* 28100 max-logit bump from §1 may be a real small sharpening at the decision position — or noise the global max picked up elsewhere. Entropy at `=` is where to adjudicate.

In [5]:
# Per-neuron W_in / W_out trajectories (one parameter_snapshot pass).
ps_epochs = np.array(variant.artifacts.get_epochs("parameter_snapshot"))
W_in_traj, W_out_traj = [], []
for e in ps_epochs:
    d = variant.artifacts.load_epoch("parameter_snapshot", int(e))
    W_in_traj.append(d["W_in"].T.astype(np.float64))   # (n_neurons, d_model)
    W_out_traj.append(d["W_out"].astype(np.float64))   # (n_neurons, d_model)
W_in_traj, W_out_traj = np.stack(W_in_traj), np.stack(W_out_traj)


def rank_exploders(eps, W, k=11):
    """Same definition as notebook 1 §1: peak event excursion in units of plateau drift."""
    plat, evt = (eps >= 20000) & (eps <= 26000), (eps >= 26500) & (eps <= 28500)
    ref = W[plat].mean(0)
    normal = np.linalg.norm(np.diff(W[plat], axis=0), axis=2).mean(0) + 1e-9
    peak = np.linalg.norm(W[evt] - ref[None], axis=2).max(0)
    return np.argsort(peak / normal)[::-1][:k]


exploders = rank_exploders(ps_epochs, W_in_traj)
print("exploders (W_in peak-excursion rank):", exploders.tolist())

exploders (W_in peak-excursion rank): [327, 400, 326, 98, 420, 303, 421, 414, 80, 63, 176]


In [6]:
# The see-saw, for the extreme exploder n327: input norm vs output norm.
win_row = np.linalg.norm(W_in_traj, axis=2)    # (E, N)
wout_row = np.linalg.norm(W_out_traj, axis=2)  # (E, N)
wout_global = np.linalg.norm(W_out_traj, axis=(1, 2))

fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=ps_epochs, y=win_row[:, 327], mode="lines",
                         name="||W_in[327]|| (input)", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=ps_epochs, y=wout_row[:, 327], mode="lines",
                         name="||W_out[327]|| (output)", line=dict(color="#d62728"), yaxis="y2"))
fig.update_layout(title="Exploder n327 see-saw: input norm vs output norm across the event",
                  xaxis_title="epoch", xaxis_range=[24000, 31000],
                  yaxis_title="||W_in[327]||", yaxis2=dict(title="||W_out[327]||",
                  overlaying="y", side="right"), legend=dict(x=0.01, y=0.99), height=440)
fig.show()

# Cohort: did the exploders' OUTPUT rows shrink relative to plateau, vs the population?
plat = (ps_epochs >= 20000) & (ps_epochs <= 26000)
evt = (ps_epochs >= 26500) & (ps_epochs <= 30000)
out_ratio = wout_row[evt].min(0) / wout_row[plat].mean(0)   # event-min / plateau-mean, per neuron
in_ratio = win_row[evt].max(0) / win_row[plat].mean(0)      # event-max / plateau-mean, per neuron
print(f"global ||W_out||_F: plateau {wout_global[plat].mean():.2f} -> event-window "
      f"[{wout_global[evt].min():.2f}, {wout_global[evt].max():.2f}]  (flat = §1 wash-out)")
print(f"population median output-shrink ratio: {np.median(out_ratio):.2f}\n")
print("neuron   ||W_in|| grow x   ||W_out|| shrink ratio")
for j in exploders:
    print(f"  {j:>3}      {in_ratio[j]:6.1f}x          {out_ratio[j]:6.3f}")

global ||W_out||_F: plateau 11.74 -> event-window [10.11, 14.84]  (flat = §1 wash-out)
population median output-shrink ratio: 0.82

neuron   ||W_in|| grow x   ||W_out|| shrink ratio
  327        43.5x           1.197
  400        22.1x           1.090
  326        25.2x           1.248
   98        10.7x           0.978
  420        15.9x           1.259
  303        11.2x           1.127
  421         7.9x           1.094
  414        10.3x           1.125
   80        10.1x           0.932
   63         8.0x           1.106
  176         8.6x           0.985


In [7]:
import torch


def attn_entropy_at_equals(epoch):
    """Per-head Shannon entropy of attention at the `=` query (index 2), mean over the grid."""
    _, cache = variant.run_with_cache(variant.make_probe(GRID), epoch=epoch)
    pat = cache["blocks.0.attn.hook_pattern"][:, :, 2, :].detach().cpu().numpy()  # (batch, heads, keys)
    ent = -(pat * np.log(np.clip(pat, 1e-12, None))).sum(-1)                       # (batch, heads)
    return ent.mean(0)                                                             # (heads,)


ent_eps = np.array([e for e in ps_epochs if 24000 <= e <= 31000])
ent = np.stack([attn_entropy_at_equals(int(e)) for e in ent_eps])  # (E, heads)

fig = go.Figure()
fig.add_vrect(x0=26900, x1=30000, fillcolor="orange", opacity=0.08, line_width=0,
              annotation_text="~27k event", annotation_position="top left")
fig.add_hline(y=float(np.log(3)), line=dict(dash="dot", color="gray"),
              annotation_text="ln 3 (uniform over a,b,=)", annotation_position="bottom right")
for h in range(ent.shape[1]):
    fig.add_trace(go.Scatter(x=ent_eps, y=ent[:, h], mode="lines", name=f"head {h}"))
fig.add_trace(go.Scatter(x=ent_eps, y=ent.mean(1), mode="lines+markers", name="mean",
                         line=dict(color="black", width=3)))
fig.update_layout(title="Attention entropy at the `=` query across the event (collapse = a dip)",
                  xaxis_title="epoch", yaxis_title="entropy (nats)", height=440)
fig.show()

base = (ent_eps >= 24000) & (ent_eps <= 26000)
mean_ent = ent.mean(1)
dip_i = int(np.argmin(mean_ent))
print(f"mean =-entropy: plateau {mean_ent[base].mean():.3f} nats -> min {mean_ent[dip_i]:.3f} "
      f"at epoch {ent_eps[dip_i]} (drop {1 - mean_ent[dip_i] / mean_ent[base].mean():.1%})")
for h in range(ent.shape[1]):
    hi = int(np.argmin(ent[:, h]))
    print(f"  head {h}: min {ent[hi, h]:.3f} at {ent_eps[hi]} (plateau {ent[base, h].mean():.3f})")

mean =-entropy: plateau 0.620 nats -> min 0.601 at epoch 29400 (drop 3.0%)
  head 0: min 0.471 at 28300 (plateau 0.530)
  head 1: min 0.663 at 27400 (plateau 0.716)
  head 2: min 0.483 at 29200 (plateau 0.525)
  head 3: min 0.689 at 24000 (plateau 0.709)


## 3. The logits across the event — does the function break and return to the *same* place?

§1–§2 read the event off the *weights* and *attention*. This pass reads it off the **output** — the logits the model actually predicts. The point of comparison is the round trip: notebook 1 found the basin "sticky in **function**, not raw coordinates" (frequencies maintained, parameters drifted). At the logit level that makes a sharp, falsifiable prediction: **the post-event logits should match the pre-event logits**, even though the weights moved — the function leaves and returns to the same place, with a genuine break in between.

Candidate checkpoints span the event (the test-loss spike peaks at ~27357, val ~3.6, but it is narrower than the 100-epoch checkpoint grid, so the highest-loss *checkpoints* sit ~27400–27600):

- **pre-event:** 26000, 26900 (clean plateau)
- **broken:** 27300, 27400, 27600 (the spike + immediate aftermath)
- **recovering / post:** 28500, 29000, 30000
- **final:** 34999

For each we read the logits at the `=` query, and report cross-entropy, grid accuracy, the per-example correct-class **margin** (`logit_correct − max_other`), and the **cosine similarity of the full logit map to the pre-event reference (26000)** — the round-trip metric. Loss/accuracy are computed straight from the forward pass, not metadata, so they are self-consistent with the logits being compared.

In [8]:
from scipy.special import logsumexp

CANDIDATES = [26000, 26900, 27300, 27400, 27600, 28500, 29000, 30000, 34999]
labels = np.array([(a + b) % PRIME for a, b in GRID])


def logits_at_equals(epoch):
    """(n_inputs, n_classes) logits read at the `=` query position."""
    logits, _ = variant.run_with_cache(variant.make_probe(GRID), epoch=epoch)
    return logits[:, -1, :].detach().cpu().numpy()


def readout(L, ref):
    """Cross-entropy, accuracy, per-example margin, cosine of the full logit map to ref."""
    ce = float(-(L[np.arange(len(labels)), labels] - logsumexp(L, axis=1)).mean())
    correct = L[np.arange(len(labels)), labels]
    other = L.copy(); other[np.arange(len(labels)), labels] = -np.inf
    margin = correct - other.max(1)
    cos = float((L.ravel() @ ref.ravel()) / (np.linalg.norm(L) * np.linalg.norm(ref)))
    return ce, float((L.argmax(1) == labels).mean()), margin, cos


L_ref = logits_at_equals(CANDIDATES[0])
rows = {e: readout(logits_at_equals(e), L_ref) for e in CANDIDATES}
print(f"{'epoch':>6} {'CE loss':>10} {'accuracy':>9} {'mean margin':>12}  cos-to-26000")
for e, (ce, acc, margin, cos) in rows.items():
    print(f"{e:>6} {ce:>10.3e} {acc:>9.3f} {margin.mean():>12.2f}    {cos:.5f}")

 epoch    CE loss  accuracy  mean margin  cos-to-26000
 26000  5.779e-09     1.000        17.07    1.00000
 26900  5.779e-09     1.000        17.07    0.99998
 27300  5.779e-09     1.000        17.07    0.99997
 27400  1.890e-03     1.000         9.80    0.98159
 27600  7.244e-03     1.000         6.16    0.99537
 28500  6.354e-07     1.000        15.18    0.98249
 29000  2.986e-08     1.000        17.11    0.98963
 30000  8.990e-09     1.000        17.07    0.99766
 34999  5.137e-09     1.000        17.10    0.99948


In [9]:
# Per-example correct-class margin distribution per candidate epoch:
# pre tight & high -> broken collapses (negatives = misclassified) -> recovered.
fig = go.Figure()
for e in CANDIDATES:
    fig.add_trace(go.Box(y=rows[e][2], name=str(e), boxpoints=False))
fig.add_hline(y=0, line=dict(dash="dot", color="gray"),
              annotation_text="margin 0 (decision boundary)", annotation_position="right")
fig.update_layout(title="Correct-class logit margin across the event (collapse at the spike, return after)",
                  xaxis_title="epoch", yaxis_title="logit_correct - max_other", height=440,
                  showlegend=False)
fig.show()

# Round-trip: how close are the recovered logits to the pre-event function?
print(f"cosine(26000, 30000) = {rows[30000][3]:.5f}   cosine(26000, 34999) = {rows[34999][3]:.5f}")
worst = min(CANDIDATES, key=lambda e: rows[e][3])
print(f"least pre-like checkpoint: {worst}  (cos {rows[worst][3]:.4f}, "
      f"acc {rows[worst][1]:.3f}, CE {rows[worst][0]:.2e})")

cosine(26000, 30000) = 0.99766   cosine(26000, 34999) = 0.99948
least pre-like checkpoint: 27400  (cos 0.9816, acc 1.000, CE 1.89e-03)


## Reading & open threads

Across all three resolutions — global weights/attention (§1), localized weights/attention (§2), and the output logits (§3) — the ~27k event matches **neither** the slingshot signature **nor** attention-entropy collapse. The literature angle does not name this event; what it does is let us rule those mechanisms out cleanly and pin what the event actually is: an **uncompensated input/activation blow-up in a few MLP rows** that briefly softens the output margin and then **round-trips back to the same function**.

**§1 — global aggregates are blind.** `||W_U||_F` peaks at epoch 2200 (grokking) and is flat (~11.5) at 27k; the global max attention logit peaks at epoch 200 (init), with only a mild ~1.3× bump at 28100, *after* the spike. The ~800× blow-up is invisible to both.

**§2 — the localized re-runs falsify the two natural mechanisms:**

- **The see-saw is one-sided — `W_out` does *not* compensate.** Exploder input rows blow up (n327 **43.5×**, n400 22×, cohort 8–44×), but their output rows do **not** shrink to offset it: per-row shrink ratios are **0.93–1.26** (exploders at/above 1.0) vs a population median of 0.82. So `a_j·W_out[j]` **overshoots** — which *is* the loss spike. **This corrects notebook 1 §7's conjecture** that `W_out` must down-scale: it doesn't; recovery comes from the input/activation blow-up **relaxing**, not output rescaling.
- **No attention-entropy collapse.** `=`-query entropy moves 0.620 → 0.601 nats (3%); the strongest head dips ~11% at 28300 — mild, head-specific, and *lagging* the spike.
- **`W_out` is the event-sensitive "last layer," `W_U` is not** — global `||W_out||_F` excurses (11.74 → [10.1, 14.8]) where `||W_U||_F` stayed flat, but it *grows* transiently rather than slingshot-snapping.

**§3 — the output round-trips to the same function:**

- **Function leaves and returns.** Cosine of the full logit map to the pre-event reference (26000) dips to **0.982** at the most-perturbed checkpoint (27400) and recovers to **0.998** (30000) / **0.9995** (34999). The model relaxes back to essentially the *same* logits despite the parameter drift — direct output-level confirmation of notebook 1's "basin sticky in **function**, not coordinates."
- **Accuracy never drops at any saved checkpoint** (1.000 throughout, including the broken 27400/27600); the break surfaces only as a **margin collapse** (mean correct-class margin 17.07 → **6.16** at 27600). The true loss-3.6 misclassification peak at **27357 falls *between* checkpoints** — the 100-epoch grid undersamples it. Saved-checkpoint *accuracy* alone would miss the event entirely: a **time-resolution** analog of §1's global→local wash-out (the signal needs finer time sampling, just as §2 needed finer unit/head sampling).

**Net read.** The ~27k event is a transient, self-healing **confidence collapse**: a handful of MLP input rows blow up, the output layer does not cancel the surplus, the correct-class margin briefly softens (instantaneously enough to spike the loss to ~3.6, but never flipping a prediction at saved resolution), and the whole configuration then relaxes back to the same function (cosine ~0.9995). Not a slingshot, not entropy collapse — uncompensated, localized, and reversible.

**Still open:**

- **Catch the true peak.** The loss-3.6 spike at 27357 has no checkpoint. If re-trained (no resume path — REQ_149), dense checkpoints across 27300–27400 would show whether accuracy *does* break at the instantaneous peak, or whether even the worst instant is margin-only.
- **Timing the overshoot.** Put `||W_in[j]||`, the contribution `a_j·W_out[j]`, the margin (§3), and the loss on one axis — does the contribution peak coincide with the margin trough?
- **Per-head max logit on the exploder-routing head**, paired with §2's per-head entropy, to finish the attention-side localization.
- **Control on p113** (clean grokker): expect flat `W_U`/max-logit, a flat see-saw, no `=`-entropy dip, and a flat logit cosine ~1.0 throughout — confirming every signal here is event-specific.
- **Promote to an analyzer?** The see-saw pair (`||W_in[j]||`, `||W_out[j]||`), `=`-entropy, and the logit-cosine round-trip are cheap per-epoch scalars — candidates beside the §7 `neuron_activation_spectrum` sketch; the see-saw and the logit-cosine are the two that *did* localize/round-trip the event.